# Select exactly what you need · Solutions

Pandas, Clearly · Pandas for Data Science · Version 1.0.0

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates

Read, predict, run, change, and explain. Every worked example starts from fresh teaching data. The original student/café tables are fictional; the public Penguins files retain their documented source.

Run the setup cell first. Keep this notebook inside the downloaded course folder so its `data/` folder can be found. Exports go to `outputs/`. For public-data lessons, extract the full notebook practice pack.

Questions: What does one row represent? Which files are inputs, and where will your outputs be written?

In [ ]:
# Pandas, Clearly
# Dr. Ali Al Bataineh
# Computer Information Science
# Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates
# Run once before the lessons. It prepares explicitly fictional practice data.
# Questions: What does 'as pd' mean? Where will exported files be saved?
# Try: print(pd.__version__) and print(OUTPUT_DIR.resolve()).
# Explain: Why should source data and analysis outputs use different folders?
import os
import tempfile
from pathlib import Path
os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "pandas-clearly-mpl"))
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import sqlite3
from pathlib import Path
from io import StringIO

# Find data from the repository root or either notebook subdirectory.
# Browser workers mount the same data tree at /course/data.
COURSE_ROOT = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents]
     if (parent / "data" / "students.csv").is_file()),
    Path.cwd(),
)
DATA_DIR = COURSE_ROOT / "data"
OUTPUT_DIR = COURSE_ROOT / "outputs"
DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

_STUDENT_ROWS = [
    ["S001", "Amira", "RAK", "AI", 20, 8, 88, 95, "2026-09-01"],
    ["S002", "Bilal", "Dubai", "IT", 21, 5, 72, 85, "2026-09-02"],
    ["S003", "Chloe", "RAK", "AI", 19, 9, 91, 98, "2026-09-03"],
    ["S004", "Dana", "Dubai", "IT", 22, 4, 65, 78, "2026-09-04"],
    ["S005", "Ethan", "RAK", "IT", 20, 6, 79, 90, "2026-09-05"],
    ["S006", "Fatima", "Dubai", "AI", 21, 10, 95, 99, "2026-09-06"],
    ["S007", "Grace", "RAK", "AI", 23, 7, 84, 92, "2026-09-07"],
    ["S008", "Hassan", "Dubai", "IT", 20, 3, 58, 70, "2026-09-08"],
    ["S009", "Ivy", "RAK", "IT", 22, 6, 76, 88, "2026-09-09"],
    ["S010", "Jamal", "Dubai", "AI", 19, 8, 89, 96, "2026-09-10"],
    ["S011", "Khalid", "RAK", "AI", 24, 5, 68, 82, "2026-09-11"],
    ["S012", "Lina", "Dubai", "IT", 21, 7, 82, 94, "2026-09-12"],
]
_STUDENT_COLUMNS = ["Student_ID", "Name", "Campus", "Program", "Age",
                    "Study_Hours", "Score", "Attendance", "Join_Date"]

def students():
    """Return a new, clean practice DataFrame, never real student records."""
    return pd.DataFrame(_STUDENT_ROWS, columns=_STUDENT_COLUMNS)

def records():
    """Return the same practice records as a list of dictionaries."""
    return students().to_dict(orient="records")

def raw_students():
    """Return a fresh table with deliberate data-quality problems."""
    df = students()
    df["Score"] = df["Score"].astype(object)
    df["Study_Hours"] = df["Study_Hours"].astype(object)
    df.loc[0, "Name"] = " Amira "
    df.loc[1, "Campus"] = "dubai"
    df.loc[3, "Score"] = None
    df.loc[5, "Study_Hours"] = None
    df.loc[7, "Score"] = "absent"
    df.loc[10, "Attendance"] = 105
    return pd.concat([df, df.iloc[[2]]], ignore_index=True)

def departments():
    return pd.DataFrame({
        "Program": ["AI", "IT"],
        "Department": ["Computing", "Information Systems"],
        "Advisor": ["Dr. Noor", "Dr. Sami"],
    })

def assessment():
    return pd.DataFrame({
        "Student_ID": ["S001", "S002", "S003", "S004", "S005", "S006",
                       "S007", "S008", "S009", "S010", "S011", "S013"],
        "Quiz": [17, 14, 19, 13, 16, 19, 17, 11, 15, 18, 14, 16],
        "Project": [44, 36, 46, 34, 40, 47, 42, 30, 38, 45, 35, 41],
    })

def attendance():
    return pd.DataFrame({
        "Date": ["2026-10-01", "2026-10-02", "2026-10-03",
                 "2026-10-05", "2026-10-06", "2026-10-07"],
        "Visitors": [20, 24, 18, 30, 28, 32],
        "Campus": ["RAK"] * 6,
    })

# Create missing practice files. Existing files are preserved.
for _name, _frame in [
    ("students.csv", students()), ("raw_students.csv", raw_students()),
    ("departments.csv", departments()), ("assessment.csv", assessment()),
    ("attendance.csv", attendance()),
]:
    if not (DATA_DIR / _name).exists():
        _frame.to_csv(DATA_DIR / _name, index=False)
if not (DATA_DIR / "students.json").exists():
    students().to_json(DATA_DIR / "students.json", orient="records", indent=2)
if not (DATA_DIR / "students.xlsx").exists():
    try:
        students().to_excel(DATA_DIR / "students.xlsx", sheet_name="Students", index=False)
    except ImportError:
        print("Excel needs openpyxl. Install it before Module 20's Excel cell.")

pd.set_option("display.max_rows", 15)
pd.set_option("display.max_columns", 12)
pd.set_option("display.width", 110)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.spines.top": False,
                     "axes.spines.right": False, "figure.dpi": 110})
print("Ready. pandas", pd.__version__)
print("Practice data: 12 fictional students. Every helper returns a fresh table.")
print("Exports go to:", OUTPUT_DIR.resolve())


## Module 05. Select exactly what you need

Choose rows, columns, and individual cells using labels and positions.

Before you start: Know the difference between a Series and a DataFrame and recognize an index.

Official reference: https://pandas.pydata.org/docs/user_guide/indexing.html

### M05C01. One bracket or two?

df["Score"] selects one column as a Series. df[["Score"]] passes a list of column names and returns a DataFrame, even with only one column. To select several columns, put their names in that inner list. The distinction matters when another operation expects a table.

**Read:** What is the inner ["Name", "Score"]?

Answer: A Python list containing the names of the columns to select.

**Predict:** What are the shapes of column and table?

Answer: (12,) for the Series and (12, 1) for the one-column DataFrame.

**Explain:** Why use brackets rather than df.Score as the main teaching pattern?

Answer: Brackets work consistently with names containing spaces and names that could conflict with DataFrame attributes or methods.



In [ ]:
# Predict before running: What are the shapes of column and table?
df = students()
column = df["Score"]
table = df[["Score"]]
result = df[["Name", "Score"]]
print(type(column).__name__, column.shape)
print(type(table).__name__, table.shape)
print(result.head(3))

**Your turn:** Select Campus and Program, in that order, into a DataFrame named result. Print its first four rows and shape.

For the check, keep these result names: `result`.

**Hint:** Place both exact column names inside one inner list.

**Watch for this:** df["Name", "Score"] is not the same as df[["Name", "Score"]] for these ordinary columns.

In [ ]:
# Change this code to complete the task.
# Question: Select Campus and Program, in that order, into a DataFrame named result. Print its first four rows and shape.
df = students()
result = df[["Campus", "Program"]]
print(result.head(4))
print(result.shape)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert isinstance(result, pd.DataFrame) and result.shape == (12, 2)
assert result.columns.tolist() == ['Campus', 'Program']
print("Code checks passed. Review the written explanation as well.")

result has 12 rows and two columns in the requested order: Campus, then Program.

**Quick check:** Which returns a one-column DataFrame?

1. df["Score"]
2. df[["Score"]]
3. df.shape

Answer: 2. Passing a list of column names preserves a DataFrame result.

**Explain what you learned:** I can predict whether a column selection returns a Series or a DataFrame.

### M05C02. Select row and column labels with loc

set_index("Student_ID") uses the student IDs as row labels and, by default, removes that field from the data columns. loc has the form table.loc[row_labels, column_labels]. A list keeps the selection table-shaped. at is convenient when you need a single value with a row label and column label.

**Read:** What separates row selection from column selection inside loc?

Answer: The comma. The row labels are on the left; the column labels are on the right.

**Predict:** Which two names appear, and what is one_score?

Answer: Amira and Chloe appear. one_score is 91.

**Explain:** Why must Student_ID be the index for this exact loc expression?

Answer: loc looks for row labels. Without set_index, the default row labels are numbers rather than S001 and S003.



In [ ]:
# Predict before running: Which two names appear, and what is one_score?
df = students().set_index("Student_ID")
result = df.loc[["S001", "S003"], ["Name", "Score"]]
one_score = df.at["S003", "Score"]
print(result)
print("One score:", one_score)

**Your turn:** Select S002 and S004 with Name and Campus into result. Use at to get S004's Score into one_score.

For the check, keep these result names: `one_score`, `result`.

**Hint:** Provide a list of IDs on the left of the comma and a list of column names on the right.

**Watch for this:** set_index does not automatically check that IDs are unique; uniqueness needs its own check.

In [ ]:
# Change this code to complete the task.
# Question: Select S002 and S004 with Name and Campus into result. Use at to get S004's Score into one_score.
df = students().set_index("Student_ID")
result = df.loc[["S002", "S004"], ["Name", "Campus"]]
one_score = df.at["S004", "Score"]
print(result)
print("One score:", one_score)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.index.tolist() == ['S002', 'S004'] and result['Campus'].tolist() == ['Dubai', 'Dubai']
assert one_score == 65
print("Code checks passed. Review the written explanation as well.")

Bilal and Dana both belong to Dubai. Dana's individual score is 65.

**Quick check:** What does loc use to select rows?

1. Only row positions.
2. Only alphabetical sorting.
3. Index labels or a Boolean mask.

Answer: 3. loc is label-based and can also accept a Boolean mask for row selection.

**Explain what you learned:** I can read selected records and one value using meaningful labels.

### M05C03. Select row and column positions with iloc

iloc uses numerical positions for both rows and columns. Positions start at 0 even when the printed index labels are student IDs. A slice excludes its stop position. iat reads one value using a row position and a column position.

**Read:** Which row positions are included by 1:4?

Answer: Positions 1, 2, and 3: Bilal, Chloe, and Dana.

**Predict:** Which columns do positions 0 and 2 represent here?

Answer: Name and Score. The table was first arranged with columns Name, Campus, Score.

**Explain:** Why can a hard-coded column position become risky after reordering columns?

Answer: The same numerical position can refer to a different column after the order changes.



In [ ]:
# Predict before running: Which columns do positions 0 and 2 represent here?
df = students()[["Name", "Campus", "Score"]]
result = df.iloc[1:4, [0, 2]]
one_score = df.iat[1, 2]
print(result)
print("One score:", one_score)

**Your turn:** Select the first two rows and all three columns into result. Read the first student's score with iat into one_score.

For the check, keep these result names: `one_score`, `result`.

**Hint:** Use :2 for the first two row positions and : for all column positions.

**Watch for this:** iloc expects positions, so df.iloc["S001"] is not valid positional selection.

In [ ]:
# Change this code to complete the task.
# Question: Select the first two rows and all three columns into result. Read the first student's score with iat into one_score.
df = students()[["Name", "Campus", "Score"]]
result = df.iloc[:2, :]
one_score = df.iat[0, 2]
print(result)
print("One score:", one_score)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert result.shape == (2, 3) and result['Name'].tolist() == ['Amira', 'Bilal']
assert one_score == 88
print("Code checks passed. Review the written explanation as well.")

result contains Amira and Bilal across all three selected columns. Amira's score is 88.

**Quick check:** What does df.iloc[1:4] include?

1. Row positions 1, 2, and 3.
2. Row positions 1, 2, 3, and 4.
3. Only a row whose label is 4.

Answer: 1. The stop position is excluded, so 1:4 includes 1, 2, and 3.

**Explain what you learned:** I can select a rectangular portion of a table by position.

### M05C04. Compare the two slice endpoints

For this sorted, unique index, a loc label slice includes both its start and stop labels. An iloc position slice excludes its stop position. Compare the selected IDs instead of assuming the two expressions return the same rows. Label slicing works best when the index is appropriately sorted and its labels are understood.

**Read:** Which expression includes the stop value?

Answer: The loc label slice includes the existing stop label S004 in this sorted unique index.

**Predict:** How many rows appear in each result?

Answer: Three in by_label: S002, S003, S004. Two in by_position: S002 and S003.

**Explain:** Does df.iloc[1:3] mean index labels 1 through 3?

Answer: No. It means row positions 1 and 2, regardless of their index labels.



In [ ]:
# Predict before running: How many rows appear in each result?
df = students().set_index("Student_ID").sort_index()
by_label = df.loc["S002":"S004", ["Name", "Score"]]
by_position = df.iloc[1:3][["Name", "Score"]]
print("Label slice:")
print(by_label)
print("Position slice:")
print(by_position)

**Your turn:** Change by_position so that its position slice returns exactly the same three rows as by_label. Store the Boolean equality check in same, then print the selected rows and the check.

For the check, keep these result names: `by_position`, `same`.

**Hint:** To include row position 3, stop at position 4.

**Watch for this:** Selecting from a selection is fine for reading; use one .loc assignment when changing the original table.

In [ ]:
# Change this code to complete the task.
# Question: Change by_position so that its position slice returns exactly the same three rows as by_label. Store the Boolean equality check in same, then print the selected rows and the check.
df = students().set_index("Student_ID").sort_index()
by_label = df.loc["S002":"S004", ["Name", "Score"]]
by_position = df.iloc[1:4][["Name", "Score"]]
same = by_label.equals(by_position)
print(by_position)
print("Same result:", same)

In [ ]:
# Check the required result. Can you explain why each condition matters?
assert same is True and by_position.index.tolist() == ['S002', 'S003', 'S004']
print("Code checks passed. Review the written explanation as well.")

Both results now contain S002, S003, and S004 with the same column order and labels. equals() reports True.

**Quick check:** On this sorted index, which is true?

1. Both loc and iloc exclude the stop.
2. loc includes its stop label; iloc excludes its stop position.
3. Both selectors ignore their start.

Answer: 2. This label slice includes S004, while a positional slice stops before its endpoint.

**Explain what you learned:** I can explain why a label slice and a position slice can return different rows.

## Keep practicing

Can you apply this idea to a different table without copying the example?

Dr. Ali Al Bataineh  
Computer Information Science  
Higher Colleges of Technology, Ras Al Khaimah, United Arab Emirates